# PrismLab P2 — N3: 10-image OCR benchmark 🧪

**Goal:** Compare **VL-only answer extraction** against **VL transcription → text-model reasoning**, using known-answer synthetic images. We will plot actual latency, answer correctness, and OCR key-phrase recall. **No benchmark measurements are pre-filled.**

**Study design:** Same image and same question for both arms. In the pipeline arm, the LFM3-1.7B model sees the **OCR text only**, not the image. The pipeline therefore does **not** increase OCR accuracy by itself; its value, if any, is in extracting or reasoning from the transcription.

> First run two images, inspect the outputs, then enable the 10-image experiment. These tests are synthetic, not a measure of real-world generalization.

## Cell 1 — Imports and locate dataset folder

In [ ]:
import sys, json, subprocess
from pathlib import Path
import pandas as pd
from IPython.display import display
from PIL import Image

here = Path.cwd().resolve()
candidates = [
    here, here.parent, here.parent.parent,
    here / "lfm3-prismlab", here.parent / "lfm3-prismlab"
]
PRISMLAB_DIR = next((p for p in candidates if (p / "benchmarks" / "ocr_benchmark.py").exists() or (p / "manifest.json").exists()), None)
if PRISMLAB_DIR is None:
    raise FileNotFoundError("Could not locate PrismLab directory.")

if (PRISMLAB_DIR / "benchmarks" / "ocr_benchmark.py").exists():
    BENCHMARKS_DIR = PRISMLAB_DIR / "benchmarks"
    DATA_DIR = PRISMLAB_DIR / "data" / "synthetic_ocr"
    RESULTS_DIR = PRISMLAB_DIR / "results" / "p2_n3_ocr_benchmark"
else:
    BENCHMARKS_DIR = PRISMLAB_DIR
    DATA_DIR = PRISMLAB_DIR
    RESULTS_DIR = PRISMLAB_DIR / "results"

ROOT = DATA_DIR
sys.path.insert(0, str(BENCHMARKS_DIR))
try:
    from ocr_benchmark import discover_model, evaluate_case, CASES
except ImportError:
    from benchmark import discover_model, evaluate_case, CASES

print("PrismLab Benchmarks:", BENCHMARKS_DIR)
print("Synthetic Dataset:", DATA_DIR)
print("Results Directory:", RESULTS_DIR)
print("Test cases:", len(CASES))


## Cell 2 — Look at the ten test images and their ground-truth questions

In [ ]:
contact_sheet = DATA_DIR / "contact_sheet.png"
if contact_sheet.exists():
    display(Image.open(contact_sheet))

display(pd.DataFrame([{
    "id": c["id"], "category": c["category"],
    "question": c["question"], "expected": c["expected_answer"]
} for c in CASES]))


## Cell 3 — Check the two local llama.cpp servers

Your VL model was on port **8180**; your text model is usually on **8080**. Modify the variables if you used other ports. A running text server is required only for the **both** comparison.

In [ ]:
VL_SERVER = "http://127.0.0.1:8180"
TEXT_SERVER = "http://127.0.0.1:8080"

VL_MODEL = discover_model(VL_SERVER)
print("Vision model:", VL_MODEL)

try:
    TEXT_MODEL = discover_model(TEXT_SERVER)
    print("Text model:", TEXT_MODEL)
    text_ready = True
except Exception as e:
    TEXT_MODEL = None
    text_ready = False
    print("Text server not connected yet:", e)

## Cell 4 — Smoke test: direct vision-answer extraction (2 images)

This uses the local model, not stored fake responses. It saves a CSV and raw JSONL under `results/`.

In [ ]:
bench_script = BENCHMARKS_DIR / "ocr_benchmark.py" if (BENCHMARKS_DIR / "ocr_benchmark.py").exists() else BENCHMARKS_DIR / "benchmark.py"
cmd = [sys.executable, str(bench_script),
       "--mode", "direct", "--limit", "2", "--tag", "smoke",
       "--vision-url", VL_SERVER]
subprocess.run(cmd, cwd=BENCHMARKS_DIR, check=True)
smoke_csv = RESULTS_DIR / "smoke_results.csv"
if not smoke_csv.exists(): smoke_csv = BENCHMARKS_DIR / "results" / "smoke_results.csv"
display(pd.read_csv(smoke_csv)[[
    "filename", "expected_answer", "direct_answer", "direct_exact", "direct_latency_s"
]])


## Cell 5 — Pipeline smoke test: 2 images, **VL-only vs VL → Text**

**Both servers must be connected.** This adds the transcription request and then asks LFM3-1.7B to extract the answer from the OCR output.

In [ ]:
if not text_ready:
    print("Start the text server on port 8080, rerun Cell 3, then run this cell again.")
else:
    bench_script = BENCHMARKS_DIR / "ocr_benchmark.py" if (BENCHMARKS_DIR / "ocr_benchmark.py").exists() else BENCHMARKS_DIR / "benchmark.py"
    cmd = [sys.executable, str(bench_script),
           "--mode", "both", "--limit", "2", "--tag", "pilot",
           "--vision-url", VL_SERVER, "--text-url", TEXT_SERVER]
    subprocess.run(cmd, cwd=BENCHMARKS_DIR, check=True)
    pilot_csv = RESULTS_DIR / "pilot_results.csv"
    if not pilot_csv.exists(): pilot_csv = BENCHMARKS_DIR / "results" / "pilot_results.csv"
    pilot = pd.read_csv(pilot_csv)
    display(pilot[["filename", "direct_answer", "pipeline_answer",
                    "direct_exact", "pipeline_exact", "ocr_phrase_recall",
                    "direct_latency_s", "pipeline_latency_s"]])


## Cell 6 — Run all 10 images

Change `RUN_FULL` to `True` **after** inspecting the pilot output. This executes up to 30 sequential model calls (10 direct + 10 OCR + 10 text) and records every measurement. Reusing the same tag replaces the previous results, so use a new tag for another pass.

In [ ]:
RUN_FULL = False  # Set to True when ready
TAG = "full_trial1"

if RUN_FULL:
    if not text_ready:
        raise RuntimeError("Text model must be running for both pipelines.")
    bench_script = BENCHMARKS_DIR / "ocr_benchmark.py" if (BENCHMARKS_DIR / "ocr_benchmark.py").exists() else BENCHMARKS_DIR / "benchmark.py"
    cmd = [sys.executable, str(bench_script),
           "--mode", "both", "--limit", "0", "--tag", TAG,
           "--vision-url", VL_SERVER, "--text-url", TEXT_SERVER]
    subprocess.run(cmd, cwd=BENCHMARKS_DIR, check=True)
    print("Saved:", RESULTS_DIR / f"{TAG}_results.csv")
else:
    print("Skipped the full run. Review the pilot, then set RUN_FULL=True.")


## Cell 7 — Plot measured comparisons

Plot the full run if it exists, otherwise the pilot. The code cannot plot data that has not been collected.

In [ ]:
from IPython.display import Image as DisplayImage

full_file = RESULTS_DIR / f"{TAG}_results.csv"
stored_full = RESULTS_DIR / "p2_n3_full_results.csv"
pilot_file = RESULTS_DIR / "pilot_results.csv"
smoke_file = RESULTS_DIR / "smoke_results.csv"
results_csv = next((p for p in [full_file, stored_full, pilot_file, smoke_file] if p.exists()), None)
if results_csv is None:
    raise FileNotFoundError("Run Cell 4 or Cell 5 before plotting.")
plot_script = BENCHMARKS_DIR / "plot_results.py"
figures_dir = PRISMLAB_DIR / "docs" / "p2" / "figures"
subprocess.run([sys.executable, str(plot_script), "--csv", str(results_csv), "--out", str(figures_dir)],
               cwd=BENCHMARKS_DIR, check=True)
for name in ["latency_comparison.png", "answer_accuracy.png", "ocr_phrase_recall.png", "resolution_ttft.png"]:
    img = figures_dir / name
    if img.exists():
        display(DisplayImage(filename=str(img)))


## Cell 8 — Inspect incorrect answers and evidence

A failed exact match may be a formatting issue, an OCR miss, or a reasoning failure. Read the raw transcription before making claims about either model.

In [ ]:
df = pd.read_csv(results_csv)
cols = [c for c in ["filename", "question", "expected_answer", "direct_answer",
                   "pipeline_answer", "direct_exact", "pipeline_exact",
                   "ocr_phrase_recall", "ocr_valid", "direct_valid", "pipeline_valid"]
        if c in df.columns]
display(df[cols])

# Inspect the OCR transcription for the first case, if it was collected
if "ocr_text" in df.columns and df["ocr_text"].notna().any():
    print("\n=== First OCR transcription ===\n", df.loc[df["ocr_text"].notna(), "ocr_text"].iloc[0])

## Research checkpoint

1. Does adding the text model improve **exact-answer correctness** on any of these images?
2. Is the added model inference latency justified by the accuracy change?
3. How does OCR key-phrase recall correlate with wrong pipeline answers?
4. Which failures are due to **image reading** versus **reasoning/extraction**?

**Limitations:** synthetic examples; one question per image; no handwritten/real-camera distribution; modest sample; no repeated-trial confidence intervals. This is an informative controlled pilot, not a benchmark paper.


## Cell 9 — N4 Experiment: Keyword-Conditioned Evidence vs. Full OCR

In addition to full OCR transcription piping, we evaluated passing only question-relevant keywords to the text model.
Below, we inspect the recorded outcomes from  and compare them against the full-pipeline results.


In [ ]:
n4_csv = RESULTS_DIR / "p2_n4_relevant_evidence.csv"
if n4_csv.exists():
    df_n4 = pd.read_csv(n4_csv)
    print(f"N4 Relevant Evidence Results ({len(df_n4)} cases):")
    print(f"Strict Exact Correct: {df_n4['correct'].sum()} / {len(df_n4)} ({df_n4['correct'].mean():.1%})")
    display(df_n4[["filename", "expected", "answer", "correct", "text_latency_s"]])
else:
    print("p2_n4_relevant_evidence.csv not found in", RESULTS_DIR)
